# PINN Landslide Model — South Mindanao (v2_8 architecture)

Trains the **same PINN v3 architecture and preprocessing as `cotabato_new_slope_unit_v2-8.ipynb`**
(`LandslideRainFallV3` + `train_model_rainfall_v3`) on **South Mindanao**
(`datasets/SlopeUnit_SouthMindanao_v3.gpkg`), then predicts and evaluates the **entire dataset**.

**Train / evaluate split (as requested):**
- **Training** uses the **undersampled** frame (all positives + `NEG_PER_POS` negatives each) — handles the
  ~0.4% imbalance and keeps runtime tractable.
- **Prediction, metrics, and maps** use the **whole** slope-filtered dataset (~1M units) via a **5-fold
  ensemble**. Rows that were in training are scored by their held-out fold (OOF), so the "held-out
  corrected" metrics are not inflated by the fact that all positives were seen during training.

**Reused from v2_8:** model, training loop, and the GA-EN selected feature set. **SM specifics:** renames +
unit conversions, `SoilType`→3-class `type`, and **separate output dirs** so the Cotabato model is untouched.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import os, json
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import tensorflow as tf

from py_files.helpers import set_seed, add_soil_texture_index
from py_files.train_rainfall_v3 import train_model_rainfall_v3, load_or_regenerate_oof

set_seed(42)

## 0. Config — paths, renames, unit conversions, feature set

In [ ]:
PROJECT_ROOT = Path.cwd().parent

INPUT_PATH = PROJECT_ROOT / "datasets" / "SlopeUnit_SouthMindanao_v3.gpkg"
LAYER = "joined_layer"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# SEPARATE dirs so Cotabato's model/manifests are never clobbered
# (train_rainfall_v3.py hardcodes 'fold-{n}-model-v3.keras' and 'v1_cotabato_transforms_fold{n}.json').
MODEL_SAVE_PATH = str(PROJECT_ROOT / "trained_models" / "south_mindanao_v2_8")
TRANSFORMS_DIR = PROJECT_ROOT / "feature_manifests" / "south_mindanao_v2_8"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(TRANSFORMS_DIR, exist_ok=True)

SLOPE_FILTER_DEG = 10.0

SM_RENAME = {
    "Slope_mean": "Slope_mean", "buk_mean": "BUK_mean", "PGA_mean": "PGA2_max",
    "Precip_mea": "Prc_mean", "CatchmentA": "ContributingFactor_mean",
    "SoilThickn": "SoilThc_mean", "Elev_mean": "Elev_mean",
    "Clay_mean": "Clay_mean", "Silt_mean": "Silt_mean", "Sand_mean": "Sand_mean",
    "SoilType": "type",
}
UNIT_CONVERSIONS = {"BUK_mean": 9.81 / 100.0, "Prc_mean": 1000.0, "SoilThc_mean": 0.001}
ZERO_IS_MISSING = {"SoilThc_mean", "BUK_mean", "Clay_mean", "Sand_mean", "Silt_mean"}

PGA_COL = "PGA2_max"
selected_numerical = [
    "Slope_mean", "BUK_mean", "Prc_mean", "ContributingFactor_mean",
    "SoilThc_mean", "soil_texture_idx", "PGA2_max", "Elev_mean",
]
selected_categorical = ["type"]
PHYSICS_FEATURES = {
    "Slope_mean", "BUK_mean", "PGA2_max",
    "Prc_mean", "ContributingFactor_mean", "SoilThc_mean",
}


def map_soil_type(value):
    """Collapse a South Mindanao soil-series name into the model's 3-class `type`
    vocabulary {Undifferentiated, Sandy Clay Loam, Loam} (texture-suffix rule)."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "Undifferentiated"
    s = str(value).lower()
    if ("undifferentiated" in s or "undefferentiated" in s
            or "mountain soil" in s or "hydrosol" in s or "peat" in s):
        return "Undifferentiated"
    if "clay" in s:
        return "Sandy Clay Loam"
    return "Loam"

## 1. Load & preprocess the WHOLE dataset (geometry kept for maps)

In [ ]:
src_cols = list(SM_RENAME.keys()) + ["landslide"]
gdf = gpd.read_file(INPUT_PATH, layer=LAYER, columns=src_cols)
print(f"Raw dataset: {len(gdf):,} rows | positives: {int(gdf['landslide'].sum()):,}")

missing_src = [c for c in SM_RENAME if c not in gdf.columns]
assert not missing_src, f"Missing source columns: {missing_src}"
gdf = gdf.rename(columns=SM_RENAME)

for col, factor in UNIT_CONVERSIONS.items():
    if col in gdf.columns and factor != 1.0:
        gdf[col] = gdf[col] * factor
        print(f"  unit-converted {col} x {factor:g}")

for col in ZERO_IS_MISSING:
    if col in gdf.columns:
        n_zero = int((gdf[col] == 0).sum())
        if n_zero:
            gdf.loc[gdf[col] == 0, col] = np.nan
            print(f"  {col}: {n_zero:,} zeros -> NaN")

gdf["type"] = gdf["type"].map(map_soil_type)
print(f"  type distribution: {gdf['type'].value_counts(dropna=False).to_dict()}")

### Slope filter -> median impute -> soil_texture_idx (same order as v2_8)
`gdf` is now the canonical WHOLE dataset. `su_id` is a stable row id used later to inject out-of-fold scores back onto the training rows.

In [ ]:
n_before = len(gdf)
gdf = gdf[gdf["Slope_mean"] >= SLOPE_FILTER_DEG].reset_index(drop=True)
print(f"Slope filter (>= {SLOPE_FILTER_DEG} deg): {n_before:,} -> {len(gdf):,} rows "
      f"(positives: {int(gdf['landslide'].sum()):,})")

impute_cols = [c for c in selected_numerical if c != "soil_texture_idx"] + ["Clay_mean", "Sand_mean", "Silt_mean"]
impute_cols = [c for c in dict.fromkeys(impute_cols) if c in gdf.columns]
medians = gdf[impute_cols].median(numeric_only=True)
imputation_medians = {c: float(medians[c]) for c in impute_cols if pd.notna(medians[c])}
gdf[impute_cols] = gdf[impute_cols].fillna(medians)

gdf = add_soil_texture_index(gdf)
assert gdf["soil_texture_idx"].notna().all()
gdf["su_id"] = np.arange(len(gdf))          # stable id; equals the (reset) index
print(f"  soil_texture_idx range: {int(gdf['soil_texture_idx'].min())}..{int(gdf['soil_texture_idx'].max())}")
print(f"  imputation medians: { {k: round(v,4) for k,v in imputation_medians.items()} }")

## 2. Build the UNDERSAMPLED training frame

The whole `gdf` is kept intact for evaluation. `gdf_train` is a class-balanced subset used **only** for
training: all positives + `NEG_PER_POS` negatives each. Set `UNDERSAMPLE=False` to train on the full set.

In [ ]:
UNDERSAMPLE = True          # train on the balanced subset (recommended for this imbalance)
NEG_PER_POS = 20

if UNDERSAMPLE:
    pos = gdf[gdf["landslide"] == 1]
    neg = gdf[gdf["landslide"] == 0].sample(
        n=min(int((gdf["landslide"] == 0).sum()), NEG_PER_POS * len(pos)), random_state=42)
    gdf_train = pd.concat([pos, neg]).sample(frac=1.0, random_state=42).reset_index(drop=True)
else:
    gdf_train = gdf.copy()

print(f"Training frame: {len(gdf_train):,} rows | positives: {int(gdf_train['landslide'].sum()):,} "
      f"({100*gdf_train['landslide'].mean():.2f}%)")
print(f"Whole dataset (for eval/maps): {len(gdf):,} rows | positives: {int(gdf['landslide'].sum()):,} "
      f"({100*gdf['landslide'].mean():.3f}%)")

## 3. Assemble geometry-free training frame

In [ ]:
feature_cols = selected_numerical + selected_categorical + ["landslide"]
missing = [c for c in feature_cols if c not in gdf_train.columns]
assert not missing, f"Missing model-input columns: {missing}"

# Geometry-free frame for training; row order matches gdf_train (su_id preserved separately).
df = pd.DataFrame(gdf_train[feature_cols]).copy()
train_su_id = gdf_train["su_id"].to_numpy()          # su_id per training row, aligned to df / oof_preds
assert df[selected_numerical].isna().sum().sum() == 0, "NaNs remain in numeric inputs"
print(f"Training frame: {df.shape} | label balance: {df['landslide'].value_counts().to_dict()}")
df.head()

## 4. Train (v2_8 pipeline, on the undersampled frame)

StratifiedKFold(5); each fold derives its own log/clip, writes a per-fold manifest to `TRANSFORMS_DIR`,
and saves `fold-{n}-model-v3.keras` to `MODEL_SAVE_PATH`. `oof_preds` are the out-of-fold scores for the
**training** rows.

In [ ]:
oof_preds, fold_aucs = train_model_rainfall_v3(
    df, selected_numerical, selected_categorical, feature_cols, PGA_COL, MODEL_SAVE_PATH,
    physics_features=PHYSICS_FEATURES, skew_threshold=1.0, clip_lower_pct=1, clip_upper_pct=99,
    transforms_dir=TRANSFORMS_DIR, categorical_encoder='embedding',
    imputed_indicator_cols=[], imputation_medians=imputation_medians,
)
print(f"\nPer-fold OOF AUCs (undersampled val): {[round(a, 4) for a in fold_aucs]}")
print(f"Mean OOF AUC: {sum(fold_aucs) / len(fold_aucs):.4f}")

## 5. Predict the WHOLE dataset (5-fold ensemble)

Every slope unit is scored by all 5 fold models (each with its own transform manifest replayed) and
averaged → `susceptibility` (the full-coverage map product). A second column `susceptibility_holdout`
replaces the training rows with their out-of-fold score, so evaluation of the positive class isn't
optimistic (all positives were in training).

In [ ]:
from tensorflow.keras.models import load_model
from py_files.GallenModel_v1 import NewmarkActivation
from py_files.data import apply_log_transform, apply_clip_thresholds, dataframe_to_dataset

fold_probs = np.zeros((len(gdf), 5), dtype=float)
for fold in range(1, 6):
    with open(TRANSFORMS_DIR / f"v1_cotabato_transforms_fold{fold}.json") as f:
        tm = json.load(f)
    m = load_model(f"{MODEL_SAVE_PATH}/fold-{fold}-model-v3.keras",
                   custom_objects={"NewmarkActivation": NewmarkActivation})
    inp = [t.name.split(":")[0] for t in m.inputs]
    fx = gdf[feature_cols].copy()
    fx = apply_log_transform(fx, tm["log_transformed_cols"])
    fx = apply_clip_thresholds(fx, tm["clip_thresholds"])
    ds = dataframe_to_dataset(fx[inp + ["landslide"]], shuffle=False, batch_size=512)
    fold_probs[:, fold - 1] = m.predict(ds)["final_head"].flatten()
    del m
    tf.keras.backend.clear_session()
    print(f"  fold {fold}: predicted {len(gdf):,} units")

gdf_eval = gdf.copy()
gdf_eval["susceptibility"] = fold_probs.mean(axis=1)          # ensemble -> maps

# Held-out correction: training rows use their OOF score (fold that didn't see them).
try:
    oof_preds
except NameError:
    oof_preds = load_or_regenerate_oof(df, feature_cols, MODEL_SAVE_PATH, transforms_dir=TRANSFORMS_DIR)
gdf_eval["susceptibility_holdout"] = gdf_eval["susceptibility"].to_numpy().copy()
gdf_eval.loc[train_su_id, "susceptibility_holdout"] = np.asarray(oof_preds, dtype=float)
print(f"OOF-corrected {len(train_su_id):,} training rows for honest metrics.")

## 6. Metrics on the WHOLE dataset

`apparent` = pure ensemble (optimistic — every positive was seen in training).
`held-out` = training rows replaced by OOF (the number to trust). Threshold selection and the confusion
matrix use the held-out scores; at ~0.4% prevalence a 0.5 cutoff is meaningless, so a best-F1 threshold is
derived from the PR curve.

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
    confusion_matrix, precision_score, recall_score, f1_score, balanced_accuracy_score,
)

y_true = gdf_eval["landslide"].to_numpy()
y_app = gdf_eval["susceptibility"].to_numpy()
y_hold = gdf_eval["susceptibility_holdout"].to_numpy()

print(f"prevalence: {y_true.mean():.4%}  (n={len(y_true):,}, positives={int(y_true.sum()):,})")
print(f"ROC AUC  — apparent: {roc_auc_score(y_true, y_app):.4f} | held-out: {roc_auc_score(y_true, y_hold):.4f}")
print(f"PR  AUC  — apparent: {average_precision_score(y_true, y_app):.4f} | "
      f"held-out: {average_precision_score(y_true, y_hold):.4f}")

roc_auc = roc_auc_score(y_true, y_hold)
pr_auc = average_precision_score(y_true, y_hold)

prec, rec, thr = precision_recall_curve(y_true, y_hold)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
THRESHOLD = float(thr[int(np.nanargmax(f1s[:-1]))])
print(f"\nBest-F1 threshold (held-out): {THRESHOLD:.4f}\n")

rows = []
for name, t in [("0.50", 0.5), (f"best-F1={THRESHOLD:.3f}", THRESHOLD)]:
    yp = (y_hold >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yp).ravel()
    rows.append({"threshold": name,
                 "precision": precision_score(y_true, yp, zero_division=0),
                 "recall": recall_score(y_true, yp, zero_division=0),
                 "f1": f1_score(y_true, yp, zero_division=0),
                 "balanced_acc": balanced_accuracy_score(y_true, yp)})
    print(f"@ {name}: TN={tn:,} FP={fp:,} FN={fn:,} TP={tp:,}")
metrics_df = pd.DataFrame(rows).set_index("threshold").round(4)
metrics_df

### 6a. ROC and Precision-Recall curves (whole dataset, held-out scores)

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_hold)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(fpr, tpr, color="#c0392b", lw=2, label=f"AUC = {roc_auc:.3f}")
ax[0].plot([0, 1], [0, 1], "--", color="gray", lw=1)
ax[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC (whole dataset)")
ax[0].legend(loc="lower right")
ax[1].plot(rec, prec, color="#2c7fb8", lw=2, label=f"AP = {pr_auc:.3f}")
ax[1].axhline(y_true.mean(), ls="--", color="gray", lw=1, label=f"baseline = {y_true.mean():.4f}")
ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (whole dataset)")
ax[1].legend(loc="upper right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "south_mindanao_train_roc_pr.png", dpi=150, bbox_inches="tight")
plt.show()

### 6b. Confusion matrix (best-F1 threshold, whole dataset)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
yp = (y_hold >= THRESHOLD).astype(int)
cm = confusion_matrix(y_true, yp)
fig, ax = plt.subplots(figsize=(4.6, 4))
ConfusionMatrixDisplay(cm, display_labels=["no LS", "LS"]).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Confusion @ thr={THRESHOLD:.3f} (whole dataset)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "south_mindanao_train_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Maps (whole dataset)

In [ ]:
import matplotlib.colors as mcolors

def plot_map(g, col, title, cbar_label, cmap="plasma_r", vmin=None, vmax=None, out=None):
    """Continuous choropleth with robust 2/98-pct scaling + best-effort basemap."""
    fig, ax = plt.subplots(figsize=(9, 8))
    vals = pd.to_numeric(g[col], errors="coerce")
    fin = vals[np.isfinite(vals)]
    lo = vmin if vmin is not None else (float(np.percentile(fin, 2)) if len(fin) else 0.0)
    hi = vmax if vmax is not None else (float(np.percentile(fin, 98)) if len(fin) else 1.0)
    if hi <= lo:
        hi = lo + 1e-6
    norm = mcolors.Normalize(lo, hi)
    g.plot(column=col, cmap=cmap, ax=ax, norm=norm)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    fig.colorbar(sm, ax=ax).set_label(cbar_label)
    ax.set_title(title); ax.set_axis_off()
    try:
        import contextily as cx
        cx.add_basemap(ax, crs=g.crs.to_string(), source=cx.providers.CartoDB.Positron)
    except Exception as exc:
        print(f"  basemap skipped for {col}: {exc}")
    fig.tight_layout()
    if out:
        fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()


def plot_categorical(g, col, colors, title, out=None):
    """Overlay categorical classes (each color drawn as its own layer)."""
    import matplotlib.patches as mpatches
    fig, ax = plt.subplots(figsize=(9, 8))
    handles = []
    for k, c in colors.items():
        sub = g[g[col] == k]
        if len(sub):
            sub.plot(ax=ax, color=c, linewidth=0)
            handles.append(mpatches.Patch(color=c, label=f"{k} ({len(sub):,})"))
    ax.legend(handles=handles, loc="lower left"); ax.set_title(title); ax.set_axis_off()
    try:
        import contextily as cx
        cx.add_basemap(ax, crs=g.crs.to_string(), source=cx.providers.CartoDB.Positron)
    except Exception as exc:
        print(f"  basemap skipped: {exc}")
    fig.tight_layout()
    if out:
        fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()

### 7a. Susceptibility (full coverage), ground truth, and confusion-class maps

In [ ]:
# Full-coverage ensemble susceptibility over every slope unit
plot_map(gdf_eval, "susceptibility", "South Mindanao — susceptibility (5-fold ensemble, whole dataset)",
         "Susceptibility", cmap="plasma_r", vmin=0.0, vmax=1.0,
         out=OUTPUT_DIR / "south_mindanao_full_susceptibility_map.png")

gdf_eval["truth"] = np.where(gdf_eval["landslide"] == 1, "landslide", "no landslide")
plot_categorical(gdf_eval, "truth", {"no landslide": "#e8e8e8", "landslide": "#d7191c"},
                 f"Ground-truth landslide units (n={int((gdf_eval['landslide']==1).sum()):,})",
                 out=OUTPUT_DIR / "south_mindanao_full_ground_truth_map.png")

yp = (gdf_eval["susceptibility_holdout"] >= THRESHOLD).astype(int)
gdf_eval["confusion"] = np.select(
    [(gdf_eval["landslide"] == 1) & (yp == 1), (gdf_eval["landslide"] == 0) & (yp == 1),
     (gdf_eval["landslide"] == 1) & (yp == 0)],
    ["TP", "FP", "FN"], default="TN")
plot_categorical(gdf_eval, "confusion",
                 {"TN": "#efefef", "FP": "#e41a1c", "FN": "#377eb8", "TP": "#4daf4a"},
                 f"Confusion classes @ thr={THRESHOLD:.3f} (whole dataset)",
                 out=OUTPUT_DIR / "south_mindanao_full_confusion_map.png")

## 8. Physics maps (best fold, whole dataset)

Intermediate geotechnical outputs (FOS, cohesion, internal friction) from the best-AUC fold model, run
over the entire dataset with that fold's transform manifest replayed.

In [ ]:
best_fold = int(np.argmax(fold_aucs)) + 1 if "fold_aucs" in dir() else 1
model_path = f"{MODEL_SAVE_PATH}/fold-{best_fold}-model-v3.keras"
with open(TRANSFORMS_DIR / f"v1_cotabato_transforms_fold{best_fold}.json") as f:
    tmeta = json.load(f)
print(f"Best fold: {best_fold}  ->  {model_path}")

bmodel = load_model(model_path, custom_objects={"NewmarkActivation": NewmarkActivation})
input_cols = [t.name.split(":")[0] for t in bmodel.inputs]

phys_df = gdf_eval[feature_cols].copy()
phys_df = apply_log_transform(phys_df, tmeta["log_transformed_cols"])
phys_df = apply_clip_thresholds(phys_df, tmeta["clip_thresholds"])
phys_ds = dataframe_to_dataset(phys_df[input_cols + ["landslide"]], shuffle=False, batch_size=512)

extractor = tf.keras.Model(inputs=bmodel.inputs, outputs={
    "fos": bmodel.get_layer("fos_layer").output,
    "cohesion": bmodel.get_layer("cohesion_layer").output,
    "internal_friction": bmodel.get_layer("internal_friction").output,
})
phys = extractor.predict(phys_ds)
for k in ("fos", "cohesion", "internal_friction"):
    gdf_eval[k] = np.asarray(phys[k]).reshape(len(gdf_eval), -1)[:, 0]
    v = gdf_eval[k]
    print(f"{k:18s} med={np.nanmedian(v):.4f} min={np.nanmin(v):.4f} max={np.nanmax(v):.4f}")

plot_map(gdf_eval, "fos", f"FOS (fold {best_fold})", "Factor of Safety", cmap="RdYlGn",
         out=OUTPUT_DIR / "south_mindanao_full_fos_map.png")
plot_map(gdf_eval, "cohesion", f"Cohesion (fold {best_fold})", "Cohesion (model units)", cmap="viridis",
         out=OUTPUT_DIR / "south_mindanao_full_cohesion_map.png")
plot_map(gdf_eval, "internal_friction", f"Internal friction (fold {best_fold})",
         "Internal friction (model units)", cmap="cividis",
         out=OUTPUT_DIR / "south_mindanao_full_internal_friction_map.png")

### 8a. (Optional) Save the full scored dataset to GeoPackage

In [ ]:
SAVE_GPKG = False   # ~1M polygons -> large file + slow write; enable when you want the vector output
if SAVE_GPKG:
    out_cols = ["su_id", "landslide", "susceptibility", "susceptibility_holdout",
                "confusion", "fos", "cohesion", "internal_friction", "geometry"]
    out_cols = [c for c in out_cols if c in gdf_eval.columns]
    out_path = OUTPUT_DIR / "south_mindanao_full_susceptibility_v2_8.gpkg"
    gdf_eval[out_cols].to_file(out_path, driver="GPKG")
    print("saved ->", out_path)
else:
    print("SAVE_GPKG=False (skipped). Flip to True to write the full scored GeoPackage.")

## Notes / caveats

- **Train vs. eval**: model trains on the undersampled `gdf_train`; all prediction/metrics/maps use the
  whole `gdf`. `susceptibility` (ensemble) is the map product; `susceptibility_holdout` (OOF on training
  rows) drives the metrics so the all-positives-in-training leakage isn't rewarded — compare `apparent` vs
  `held-out` AUC in §6.
- **Judge by ROC AUC / PR AUC** (not accuracy) at ~0.4% prevalence; the best-F1 threshold trades precision
  for recall — adjust `THRESHOLD` for your operational point.
- **Outputs** -> `outputs/south_mindanao_full_*.png` (+ optional `.gpkg` in §8a).
- **Dirs**: checkpoints in `trained_models/south_mindanao_v2_8/`, manifests in
  `feature_manifests/south_mindanao_v2_8/` (hardcoded `v1_cotabato`/`fold-*-v3` filename stems).